# 01 — Environment & Hardware Check (Google Colab Free T4)
# Kiểm tra Môi trường & Phần cứng (Google Colab T4 Miễn phí)

**English:**
This notebook verifies your Google Colab Free T4 runtime, Python/CUDA/PyTorch versions, Hugging Face ML library compatibility (`transformers`, `peft`, `trl`, `bitsandbytes`, `accelerate`, `datasets`), Google Drive persistence (`/content/drive/MyDrive/mimamori_companion_ai`), and repository paths **without downloading multi-GB model weights**.

**Tiếng Việt:**
Notebook này kiểm tra môi trường Google Colab T4 miễn phí, phiên bản Python/CUDA/PyTorch, khả năng tương thích các thư viện huấn luyện (`transformers`, `peft`, `trl`, `bitsandbytes`, `accelerate`, `datasets`), kết nối lưu trữ bền vững trên Google Drive và đường dẫn file trong `ai-training/` **mà chưa tải trọng số mô hình nhiều GB**.

> **Free Tier & Disconnect Safeguards / Lưu ý an toàn Colab miễn phí:**
> - **Use ONLY free T4 GPU (`15 GB VRAM`, Turing `sm_75`)** (`Runtime -> Change runtime type -> T4 GPU`). **NEVER** enable paid GPUs or billing.
> - Always enforce `fp16=True, bf16=False` on Tesla T4 (Turing lacks native `bf16` tensor cores & FlashAttention-2).
> - All processed splits, checkpoints, adapters, and evaluation results are persisted to `/content/drive/MyDrive/mimamori_companion_ai` so work survives Colab disconnects.


## Step 1: Mount Google Drive & Locate `ai-training/` Configs
## Bước 1: Kết nối Google Drive & Định vị Thư mục `ai-training/`

Run this cell first after every Colab connect/reconnect. Set `CUSTOM_AI_TRAINING_DIR` (or env var `MIMAMORI_AI_TRAINING_DIR`) if your repo is cloned in a non-default path.


In [ ]:
import os
import sys
import shutil
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
DRIVE_ROOT_DEFAULT = Path("/content/drive/MyDrive/mimamori_companion_ai")

# Optional: set AI_TRAINING_DIR explicitly if your repo lives in a custom folder
CUSTOM_AI_TRAINING_DIR = os.environ.get("MIMAMORI_AI_TRAINING_DIR", "").strip()

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    print("[Drive] Google Drive mounted at /content/drive")
else:
    print("[Local] Not running inside Google Colab; Drive mount skipped.")


def resolve_ai_training_dir() -> Path:
    candidates: list[Path] = []
    if CUSTOM_AI_TRAINING_DIR:
        candidates.append(Path(CUSTOM_AI_TRAINING_DIR))
    cwd = Path.cwd().resolve()
    candidates.extend([
        cwd,
        cwd.parent,
        cwd / "ai-training",
        cwd.parent / "ai-training",
        Path("/content/mimamori/ai-training"),
        Path("/content/ai-training"),
        Path("/content/drive/MyDrive/mimamori/ai-training"),
        Path("/content/drive/MyDrive/mimamori_companion_ai/repo/ai-training"),
        Path("/content/drive/MyDrive/mimamori_companion_ai/ai-training"),
    ])
    for cand in candidates:
        if (cand / "configs" / "model_config.yaml").exists() and (cand / "configs" / "training_config.yaml").exists():
            return cand.resolve()
    raise FileNotFoundError(
        "Could not locate 'ai-training/configs/model_config.yaml'. "
        "In Colab, clone/upload the repository to /content/mimamori or Google Drive, "
        "or set CUSTOM_AI_TRAINING_DIR / MIMAMORI_AI_TRAINING_DIR."
    )


AI_TRAINING_DIR = resolve_ai_training_dir()
if str(AI_TRAINING_DIR / "scripts") not in sys.path:
    sys.path.insert(0, str(AI_TRAINING_DIR / "scripts"))

import yaml

MODEL_CONFIG_PATH = AI_TRAINING_DIR / "configs" / "model_config.yaml"
TRAINING_CONFIG_PATH = AI_TRAINING_DIR / "configs" / "training_config.yaml"
with MODEL_CONFIG_PATH.open("r", encoding="utf-8") as f:
    MODEL_CFG = yaml.safe_load(f)
with TRAINING_CONFIG_PATH.open("r", encoding="utf-8") as f:
    TRAIN_CFG = yaml.safe_load(f)

PATHS_CFG = TRAIN_CFG["paths"]
if IN_COLAB:
    DRIVE_ROOT = Path(PATHS_CFG["drive_root"])
    DRIVE_CHECKPOINTS_DIR = Path(PATHS_CFG["drive_checkpoints_dir"])
    DRIVE_FINAL_ADAPTER_DIR = Path(PATHS_CFG["drive_final_adapter_dir"])
    DRIVE_EVAL_OUTPUTS_DIR = Path(PATHS_CFG["drive_eval_outputs_dir"])
    DRIVE_PROCESSED_DATA_DIR = Path(PATHS_CFG["drive_processed_data_dir"])
else:
    local_drive_sim = AI_TRAINING_DIR / "outputs" / "_local_drive_mirror"
    DRIVE_ROOT = local_drive_sim
    DRIVE_CHECKPOINTS_DIR = local_drive_sim / "checkpoints"
    DRIVE_FINAL_ADAPTER_DIR = local_drive_sim / "adapters" / "qwen3-4b-companion-lora"
    DRIVE_EVAL_OUTPUTS_DIR = AI_TRAINING_DIR / PATHS_CFG["local_outputs_dir"]
    DRIVE_PROCESSED_DATA_DIR = AI_TRAINING_DIR / "datasets" / "processed"

for d in (DRIVE_ROOT, DRIVE_CHECKPOINTS_DIR, DRIVE_FINAL_ADAPTER_DIR, DRIVE_EVAL_OUTPUTS_DIR, DRIVE_PROCESSED_DATA_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"[Repo] AI_TRAINING_DIR      = {AI_TRAINING_DIR}")
print(f"[Drive] DRIVE_ROOT          = {DRIVE_ROOT}")
print(f"[Drive] DRIVE_EVAL_OUTPUTS  = {DRIVE_EVAL_OUTPUTS_DIR}")
print(f"[Drive] DRIVE_PROCESSED     = {DRIVE_PROCESSED_DATA_DIR}")


## Step 2: Check GPU, CUDA, VRAM & Disk/RAM (Free T4 15GB Safeguards)
## Bước 2: Kiểm tra GPU, CUDA, VRAM & Dung lượng Đĩa/RAM

Verifies `torch.cuda.get_device_properties(0)`, Turing `sm_75` compute capability, and previews whether standard T4 settings (`batch_size=2, grad_accum=8, max_seq_length=1024`) or the OOM-safe fallback (`batch_size=1, grad_accum=16, max_seq_length=768`) will be selected.


In [ ]:
import json
import platform

print(f"Python version : {sys.version.split()[0]} ({platform.platform()})")
if sys.version_info < (3, 11):
    print("[WARNING] Python >= 3.11 is required by latest transformers/peft/trl/accelerate on PyPI.")

disk_content = shutil.disk_usage("/content" if Path("/content").exists() else str(AI_TRAINING_DIR))
print(f"Workspace disk free: {disk_content.free / (1024**3):.2f} GB / {disk_content.total / (1024**3):.2f} GB")

if Path("/content/drive/MyDrive").exists():
    disk_drive = shutil.disk_usage("/content/drive/MyDrive")
    print(f"Google Drive free  : {disk_drive.free / (1024**3):.2f} GB")

try:
    import torch
    print(f"PyTorch version: {torch.__version__} | CUDA built: {torch.version.cuda} | CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        print(f"GPU Name       : {props.name}")
        print(f"Total VRAM     : {props.total_memory / (1024**3):.2f} GB")
        print(f"SM Capability  : {props.major}.{props.minor} (multi_processor_count={props.multi_processor_count})")
except ImportError as exc:
    print(f"[MISSING] torch not installed: {exc}")


def inspect_gpu_and_t4_safeguards(train_cfg: dict) -> dict:
    """Detect GPU/VRAM and return Free Colab T4 15GB safe hyperparameters."""
    t4_target = train_cfg["hardware_target"]
    base_train = dict(train_cfg["training"])
    fallback = dict(train_cfg["t4_oom_fallback"])
    info = {
        "cuda_available": False,
        "gpu_name": "CPU",
        "vram_gb": 0.0,
        "compute_capability": None,
        "supports_bf16": False,
        "supports_flash_attention_2": False,
        "use_t4_oom_fallback": False,
        "effective_fp16": True,
        "effective_bf16": False,
        "effective_batch_size": fallback["per_device_train_batch_size"],
        "effective_grad_accum": fallback["gradient_accumulation_steps"],
        "effective_max_seq_length": fallback["max_seq_length"],
        "warnings": [],
    }

    try:
        import torch
    except ImportError:
        info["warnings"].append("PyTorch is not installed; GPU checks unavailable.")
        return info

    if not torch.cuda.is_available():
        info["warnings"].append(
            "No CUDA GPU detected. In Colab, select Runtime -> Change runtime type -> T4 GPU (Free tier; NEVER enable paid GPU/billing)."
        )
        return info

    props = torch.cuda.get_device_properties(0)
    vram_gb = round(props.total_memory / (1024 ** 3), 2)
    major, minor = torch.cuda.get_device_capability(0)
    supports_ampere = (major, minor) >= (8, 0)

    info["cuda_available"] = True
    info["gpu_name"] = props.name
    info["vram_gb"] = vram_gb
    info["compute_capability"] = f"{major}.{minor}"
    # Enforce T4 safe precision policy per hardware_target (fp16=True, bf16=False on Colab T4)
    info["supports_bf16"] = bool(t4_target.get("supports_bf16", False)) and supports_ampere
    info["supports_flash_attention_2"] = bool(t4_target.get("supports_flash_attention_2", False)) and supports_ampere

    if vram_gb <= 15.5:
        info["warnings"].append(
            f"Detected <=15.5GB GPU ({props.name}, {vram_gb} GB, sm_{major}{minor}). "
            "Applying Free T4 safeguards: fp16=True, bf16=False, gradient_checkpointing=True, paged_adamw_8bit."
        )
    if vram_gb < 14.5:
        info["use_t4_oom_fallback"] = True
        info["effective_batch_size"] = fallback["batch_size"]
        info["effective_grad_accum"] = fallback["gradient_accumulation_steps"]
        info["effective_max_seq_length"] = fallback["max_seq_length"]
        info["warnings"].append(
            f"VRAM ({vram_gb} GB) < 14.5 GB -> auto-switching to t4_oom_fallback: "
            f"batch_size={fallback['batch_size']}, grad_accum={fallback['gradient_accumulation_steps']}, "
            f"max_seq_length={fallback['max_seq_length']}."
        )
    else:
        info["effective_batch_size"] = base_train["per_device_train_batch_size"]
        info["effective_grad_accum"] = base_train["gradient_accumulation_steps"]
        info["effective_max_seq_length"] = base_train["max_seq_length"]

    if not supports_ampere:
        info["effective_fp16"] = True
        info["effective_bf16"] = False
    return info


GPU_SAFEGUARD = inspect_gpu_and_t4_safeguards(TRAIN_CFG)
for _w in GPU_SAFEGUARD["warnings"]:
    print(f"[T4 Safeguard] {_w}")
print("[Hardware Profile]", GPU_SAFEGUARD)


## Step 3: Check Library Versions & Model Compatibility Matrix
## Bước 3: Kiểm tra Phiên bản Thư viện & Tương thích Mô hình

Thresholds verified in `configs/model_config.yaml`:
- **`Qwen/Qwen3-4B-Instruct-2507`** (`primary_model`, `qwen3` / `Qwen3ForCausalLM`): requires `transformers >= 4.51.0` (recommended `>= 4.57.6`), `peft >= 0.14.0`, `trl >= 0.16.0`, `bitsandbytes >= 0.45.0`, `accelerate >= 1.5.0`, `datasets >= 3.2.0`.
- **`Qwen/Qwen3.5-4B`** (`comparison_model`, `qwen3_5` / `Qwen3_5ForConditionalGeneration`): strictly requires `transformers >= 5.2.0` + `enable_thinking=False` (and `torchvision`/`pillow`).


In [ ]:
import importlib
import importlib.metadata
import re


def parse_ver(ver_str: str) -> tuple[int, int, int]:
    m = re.match(r"^\s*(\d+)\.(\d+)(?:\.(\d+))?", str(ver_str))
    return (int(m.group(1)), int(m.group(2)), int(m.group(3) or 0)) if m else (0, 0, 0)


DIST_NAME_MAP = {"PIL": "pillow", "yaml": "pyyaml"}


def check_pkg(pkg_name: str, min_ver: str | None = None, rec_ver: str | None = None) -> dict:
    dist_name = DIST_NAME_MAP.get(pkg_name, pkg_name)
    try:
        mod = importlib.import_module(pkg_name)
        ver = getattr(mod, "__version__", None)
        if not ver:
            try:
                ver = importlib.metadata.version(dist_name)
            except importlib.metadata.PackageNotFoundError:
                ver = "unknown"
        installed = True
    except Exception:
        ver = None
        installed = False

    meets_min = bool(installed and min_ver and parse_ver(ver) >= parse_ver(min_ver)) if min_ver else installed
    meets_rec = bool(installed and rec_ver and parse_ver(ver) >= parse_ver(rec_ver)) if rec_ver else meets_min
    return {
        "package": pkg_name,
        "installed": installed,
        "version": ver,
        "min_required": min_ver,
        "recommended": rec_ver,
        "meets_min": meets_min,
        "meets_recommended": meets_rec,
    }


qwen3_min_tf = MODEL_CFG["primary_model"]["min_transformers_version"]
qwen3_rec_tf = MODEL_CFG["primary_model"]["recommended_transformers_version"]
qwen35_min_tf = MODEL_CFG["comparison_model"]["min_transformers_version"]

checks = [
    check_pkg("transformers", qwen3_min_tf, qwen3_rec_tf),
    check_pkg("peft", "0.14.0", "0.18.0"),
    check_pkg("trl", "0.16.0", "0.22.2"),
    check_pkg("bitsandbytes", "0.45.0", "0.45.5"),
    check_pkg("accelerate", "1.5.0", "1.10.0"),
    check_pkg("datasets", "3.2.0", "4.0.0"),
    check_pkg("yaml", None, None),
    check_pkg("huggingface_hub", None, None),
    check_pkg("torchvision", None, None),
    check_pkg("PIL", None, None),
]

for row in checks:
    status = "OK" if row["meets_min"] else ("INSTALLED_OLD" if row["installed"] else "MISSING")
    print(f"{row['package']:16s} | {status:13s} | version={str(row['version']):10s} | min={row['min_required']} | rec={row['recommended']}")

tf_ver = checks[0]["version"]
qwen3_ok = bool(tf_ver and parse_ver(tf_ver) >= parse_ver(qwen3_min_tf))
qwen35_ok = bool(tf_ver and parse_ver(tf_ver) >= parse_ver(qwen35_min_tf))

print("\n--- Model Architecture Gate ---")
print(f"Qwen/Qwen3-4B-Instruct-2507 (transformers >= {qwen3_min_tf}): {'SUPPORTED' if qwen3_ok else 'NOT_SUPPORTED'}")
print(f"Qwen/Qwen3.5-4B             (transformers >= {qwen35_min_tf}): {'SUPPORTED' if qwen35_ok else 'NOT_SUPPORTED (upgrade transformers>=5.2.0 only if testing Qwen3.5)'}")


## Step 4 (Optional in Colab): Install / Upgrade Missing Training Packages
## Bước 4 (Tùy chọn trên Colab): Cài đặt / Nâng cấp Thư viện Thiếu

Keep `INSTALL_COLAB_DEPS = False` locally or when Colab already meets the minimum versions. Flip to `True` on Google Colab if any required library is missing (`bitsandbytes`, `trl`, `peft`, `datasets`). Optionally set `UPGRADE_FOR_QWEN35 = True` ONLY if you want `transformers >= 5.2.0` to evaluate `Qwen/Qwen3.5-4B`.


In [ ]:
import subprocess

INSTALL_COLAB_DEPS = False
UPGRADE_FOR_QWEN35 = False

if INSTALL_COLAB_DEPS:
    if not IN_COLAB:
        print("[Skip] INSTALL_COLAB_DEPS is intended for Google Colab; skipping on local machine.")
    else:
        tf_spec = "transformers>=5.2.0" if UPGRADE_FOR_QWEN35 else "transformers>=4.57.6,<5.0.0"
        pkgs = [
            tf_spec,
            "peft>=0.18.0",
            "trl>=0.22.2",
            "bitsandbytes>=0.45.5",
            "accelerate>=1.10.0",
            "datasets>=4.0.0",
            "pyyaml>=6.0",
        ]
        if UPGRADE_FOR_QWEN35:
            pkgs.extend(["torchvision", "pillow"])
        print("Installing:", pkgs)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *pkgs])
        print("[Done] Packages installed. Restart Colab runtime if transformers was upgraded in-place.")
else:
    print("[Info] INSTALL_COLAB_DEPS=False -> no pip changes performed.")


## Step 5: Verify Repo Files, Configs & Optional Colab `GROQ_API_KEY` Secret (No Weight Downloads)
## Bước 5: Xác nhận File Dự án, Cấu hình & Colab Secret (Không Tải Trọng số)

Checks that all scripts, configs, and raw/eval datasets exist, and verifies whether `GROQ_API_KEY` is present in Colab Secrets (`userdata.get`) or environment variables **without ever printing or saving the key**.


In [ ]:
expected_files = [
    AI_TRAINING_DIR / "configs" / "model_config.yaml",
    AI_TRAINING_DIR / "configs" / "training_config.yaml",
    AI_TRAINING_DIR / PATHS_CFG["local_raw_seed_file"],
    AI_TRAINING_DIR / PATHS_CFG["local_eval_benchmark_file"],
    AI_TRAINING_DIR / "scripts" / "prepare_dataset.py",
    AI_TRAINING_DIR / "scripts" / "validate_dataset.py",
    AI_TRAINING_DIR / "scripts" / "evaluate_model.py",
]

missing_files = [str(p) for p in expected_files if not p.exists()]
for p in expected_files:
    state = "OK" if p.exists() else "MISSING"
    print(f"[{state:7s}] {p.relative_to(AI_TRAINING_DIR)}")

if missing_files:
    raise FileNotFoundError(f"Missing required files: {missing_files}")

# Check GROQ_API_KEY safely (no secret printing)
groq_env_name = MODEL_CFG["fallback_llm"]["api_key_env"]
groq_present = bool(os.environ.get(groq_env_name, "").strip())
if not groq_present and IN_COLAB:
    try:
        from google.colab import userdata
        secret_val = userdata.get(groq_env_name)
        groq_present = bool(secret_val and secret_val.strip())
    except Exception:
        groq_present = False

print(f"\n[Secret Check] {groq_env_name} available: {groq_present} (optional; only needed if running Groq baseline in notebook 03)")
print("[Primary Model Config]", json.dumps({
    "id": MODEL_CFG["primary_model"]["id"],
    "arch": MODEL_CFG["primary_model"]["arch"],
    "enable_thinking": MODEL_CFG["primary_model"]["enable_thinking"],
    "lora_target_modules": MODEL_CFG["primary_model"]["lora_target_modules"],
    "quantization": MODEL_CFG["primary_model"]["quantization"],
}, ensure_ascii=False, indent=2))
print("[Note] No model weights were downloaded in 01_environment_check.ipynb. Proceed to 02_dataset_preparation.ipynb.")
